# Silver Layer - Transformation Pipeline

Rules applied per source:
- No business logic - only cleaning, typing, filtering and conforming
- Rejected rows logged separately, never silently dropped
- Output: one clean Parquet per source, ready for Gold join

## Sources
1. bronze_unemployment.parquet → silver_unemployment.parquet
2. bronze_cpi.parquet → silver_cpi.parquet
3. bronze_policy_rate.parquet → silver_policy_rate.parquet

In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

BRONZE_PATH = Path("../data/bronze")
SILVER_PATH = Path("../data/silver")
SILVER_PATH.mkdir(parents=True, exist_ok=True)

In [2]:
# Load Bronze files
df_bronze_unemployment = pd.read_parquet(BRONZE_PATH / "bronze_unemployment.parquet")
df_bronze_cpi = pd.read_parquet(BRONZE_PATH / "bronze_cpi.parquet")
df_bronze_policy = pd.read_parquet(BRONZE_PATH / "bronze_policy_rate.parquet")

print(f"Bronze unemployment : {df_bronze_unemployment.shape}")
print(f"Bronze CPI          : {df_bronze_cpi.shape}")
print(f"Bronze policy rate  : {df_bronze_policy.shape}")

Bronze unemployment : (9888, 18)
Bronze CPI          : (2518, 8)
Bronze policy rate  : (2710, 8)


## Transform 1 - OFS Unemployment
Filters applied:
- FREQ = M (monthly only)
- PERIOD >= 2010-M01 (remove null period)
- GENDER_FR = Total (aggregate, no sex breakdown)
- INDICATORS_FR = NUTS2 (regional level only)

Columns: drop DE duplicates, rename for clarity, parse PERIOD to YYYY-MM

In [3]:
def transform_unemployment(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Transform bronze unemployment data to silver.
    Returns (accepted, rejected) DataFrames.
    """
    rejected_rows = []

    # ── Filter FREQ = M ──────────────────────────────────────────────────────
    mask_freq = df["FREQ"] == "M"
    rejected_rows.append(df[~mask_freq].assign(_rejection_reason="FREQ != M"))
    df = df[mask_freq].copy()

    # ── Filter PERIOD >= 2010-M01 ────────────────────────────────────────────
    mask_period = df["PERIOD"] >= "2010-M01"
    rejected_rows.append(df[~mask_period].assign(_rejection_reason="PERIOD < 2010-M01"))
    df = df[mask_period].copy()

    # ── Filter GENDER_FR = Total ─────────────────────────────────────────────
    mask_gender = df["GENDER_FR"] == "Total"
    rejected_rows.append(df[~mask_gender].assign(_rejection_reason="GENDER_FR != Total"))
    df = df[mask_gender].copy()

    # ── Filter INDICATORS_FR = NUTS2 ─────────────────────────────────────────
    mask_nuts = df["INDICATORS_FR"] == "NUTS2"
    rejected_rows.append(df[~mask_nuts].assign(_rejection_reason="INDICATORS_FR != NUTS2"))
    df = df[mask_nuts].copy()

    # ── Parse PERIOD: 2010-M01 → 2010-01 ────────────────────────────────────
    df["period"] = df["PERIOD"].str.replace("-M", "-", regex=False)

    # ── Drop DE columns + unused columns ────────────────────────────────────
    cols_to_drop = ["INDICATORS_HRCHY", "INDICATORS_DE", "GENDER_DE",
                    "DETAILS_DE", "FREQ", "MEASURE_FR", "MEASURE_DE",
                    "STATUS", "STATUS_1", "PERIOD"]
    df = df.drop(columns=cols_to_drop)

    # ── Rename columns ───────────────────────────────────────────────────────
    df = df.rename(columns={
        "INDICATORS_FR" : "indicator",
        "GENDER_FR"     : "gender",
        "DETAILS_FR"    : "region",
        "VALUE"         : "unemployment_rate"
    })

    rejected = pd.concat(rejected_rows, ignore_index=True) if rejected_rows else pd.DataFrame()

    return df, rejected


silver_unemployment, rejected_unemployment = transform_unemployment(df_bronze_unemployment)

print(f"Accepted : {len(silver_unemployment)} rows")
print(f"Rejected : {len(rejected_unemployment)} rows")
print(f"\nSample:\n{silver_unemployment.head(3)}")
print(f"\nRejection reasons:\n{rejected_unemployment['_rejection_reason'].value_counts()}")

Accepted : 1365 rows
Rejected : 8523 rows

Sample:
     indicator gender                region  unemployment_rate  \
2305     NUTS2  Total      Région lémanique           7.887477   
2306     NUTS2  Total     Espace Mittelland           5.418161   
2307     NUTS2  Total  Suisse du Nord-Ouest           4.946085   

                                   _run_id                      _ingested_at  \
2305  88455a82-f70b-4392-b459-1c2e7a0f0dac  2026-07-15T12:59:47.583578+00:00   
2306  88455a82-f70b-4392-b459-1c2e7a0f0dac  2026-07-15T12:59:47.583578+00:00   
2307  88455a82-f70b-4392-b459-1c2e7a0f0dac  2026-07-15T12:59:47.583578+00:00   

                   _source                                        _source_url  \
2305  ofs_unemployment_bit  https://dam-api.bfs.admin.ch/hub/api/dam/asset...   
2306  ofs_unemployment_bit  https://dam-api.bfs.admin.ch/hub/api/dam/asset...   
2307  ofs_unemployment_bit  https://dam-api.bfs.admin.ch/hub/api/dam/asset...   

       period  
2305  2010-01  
2306  

In [4]:
# Save silver unemployment
silver_unemployment.to_parquet(SILVER_PATH / "silver_unemployment.parquet", index=False)
rejected_unemployment.to_parquet(SILVER_PATH / "silver_unemployment_rejected.parquet", index=False)

print(f"Saved silver_unemployment        : {len(silver_unemployment)} rows")
print(f"Saved silver_unemployment_rejected: {len(rejected_unemployment)} rows")

Saved silver_unemployment        : 1365 rows
Saved silver_unemployment_rejected: 8523 rows


## Transform 2 - SNB CPI
Filters applied:
- date >= 2010-01
- Keep both series: LD2010100 (index) and VVP (YoY inflation rate)

Columns: rename for clarity, cast date to string YYYY-MM (already correct format)

In [5]:
def transform_cpi(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Transform bronze CPI data to silver.
    Returns (accepted, rejected) DataFrames.
    """
    rejected_rows = []

    # ── Filter date >= 2010-01 ───────────────────────────────────────────────
    mask_date = df["date"] >= "2010-01"
    rejected_rows.append(df[~mask_date].assign(_rejection_reason="date < 2010-01"))
    df = df[mask_date].copy()

    # ── Extract series key short name ────────────────────────────────────────
    df["series_key"] = df["_series_key"].str.extract(r'\{(\w+)\}')

    # ── Rename columns ───────────────────────────────────────────────────────
    df = df.rename(columns={
        "date"  : "period",
        "value" : "cpi_value"
    })

    # ── Drop redundant columns ───────────────────────────────────────────────
    df = df.drop(columns=["_series_key", "_series_name"])

    rejected = pd.concat(rejected_rows, ignore_index=True) if rejected_rows else pd.DataFrame()

    return df, rejected


silver_cpi, rejected_cpi = transform_cpi(df_bronze_cpi)

print(f"Accepted : {len(silver_cpi)} rows")
print(f"Rejected : {len(rejected_cpi)} rows")
print(f"\nSeries breakdown:\n{silver_cpi.groupby('series_key')['period'].agg(['min', 'max', 'count'])}")
print(f"\nSample:\n{silver_cpi.head(4)}")

Accepted : 394 rows
Rejected : 2124 rows

Series breakdown:
                min      max  count
series_key                         
LD2010100   2010-01  2026-05    197
VVP         2010-01  2026-05    197

Sample:
       period  cpi_value                               _run_id  \
1068  2010-01    94.6843  88455a82-f70b-4392-b459-1c2e7a0f0dac   
1069  2010-02    94.8221  88455a82-f70b-4392-b459-1c2e7a0f0dac   
1070  2010-03    94.9521  88455a82-f70b-4392-b459-1c2e7a0f0dac   
1071  2010-04    95.7637  88455a82-f70b-4392-b459-1c2e7a0f0dac   

                          _ingested_at  _source  \
1068  2026-07-15T12:59:47.583578+00:00  snb_cpi   
1069  2026-07-15T12:59:47.583578+00:00  snb_cpi   
1070  2026-07-15T12:59:47.583578+00:00  snb_cpi   
1071  2026-07-15T12:59:47.583578+00:00  snb_cpi   

                                           _source_url series_key  
1068  https://data.snb.ch/api/cube/plkopr/data/json/fr  LD2010100  
1069  https://data.snb.ch/api/cube/plkopr/data/json/fr  LD201010

In [6]:
silver_cpi.to_parquet(SILVER_PATH / "silver_cpi.parquet", index=False)
rejected_cpi.to_parquet(SILVER_PATH / "silver_cpi_rejected.parquet", index=False)

print(f"Saved silver_cpi        : {len(silver_cpi)} rows")
print(f"Saved silver_cpi_rejected: {len(rejected_cpi)} rows")

Saved silver_cpi        : 394 rows
Saved silver_cpi_rejected: 2124 rows


## Transform 3 - SNB Policy Rate
Logic:
- Keep Swiss series only (drop Fed, ECB, BoE, BoJ)
- Reconstruct continuous Swiss rate 2010-2026:
  - 2010-01 to 2019-05: midpoint of LIBOR range (lower + upper) / 2
  - 2019-06 to present: SNB policy rate directly
- Output: one row per period with unified swiss_policy_rate column

In [8]:
def transform_policy_rate(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Transform bronze policy rate to silver.
    Reconstructs a continuous Swiss rate series 2010-2026.
    Returns (accepted, rejected) DataFrames.
    """
    rejected_rows = []

    # ── Keep Swiss series only, reject others ────────────────────────────────
    swiss_mask = df["_series_name"].str.startswith("Suisse")
    rejected_rows.append(df[~swiss_mask].assign(_rejection_reason="Non-Swiss central bank"))
    df_swiss = df[swiss_mask].copy()

    # ── Split into 3 series ──────────────────────────────────────────────────
    df_rate    = df_swiss[df_swiss["_series_name"].str.contains("Taux directeur de la BNS")]
    df_lower   = df_swiss[df_swiss["_series_name"].str.contains("limite inférieure")]
    df_upper   = df_swiss[df_swiss["_series_name"].str.contains("limite supérieure")]

    # ── Reconstruct 2010-2019: LIBOR midpoint ────────────────────────────────
    df_libor = df_lower[["date", "value", "_run_id", "_ingested_at", "_source", "_source_url"]].copy()
    df_libor = df_libor.merge(
        df_upper[["date", "value"]].rename(columns={"value": "value_upper"}),
        on="date"
    )
    df_libor["swiss_policy_rate"] = (df_libor["value"] + df_libor["value_upper"]) / 2
    df_libor = df_libor[(df_libor["date"] >= "2010-01") & (df_libor["date"] < "2019-06")]
    df_libor = df_libor[["date", "swiss_policy_rate", "_run_id", "_ingested_at", "_source", "_source_url"]]
    df_libor["_rate_source"] = "LIBOR_midpoint"

    # ── 2019-2026: SNB policy rate ───────────────────────────────────────────
    df_snb = df_rate[df_rate["date"] >= "2019-06"][["date", "value", "_run_id", "_ingested_at", "_source", "_source_url"]].copy()
    df_snb = df_snb.rename(columns={"value": "swiss_policy_rate"})
    df_snb["_rate_source"] = "SNB_policy_rate"

    # ── Concatenate both periods ─────────────────────────────────────────────
    df_final = pd.concat([df_libor, df_snb], ignore_index=True)
    df_final = df_final.rename(columns={"date": "period"})
    df_final = df_final.sort_values("period").reset_index(drop=True)

    rejected = pd.concat(rejected_rows, ignore_index=True) if rejected_rows else pd.DataFrame()

    return df_final, rejected


silver_policy, rejected_policy = transform_policy_rate(df_bronze_policy)

print(f"Accepted : {len(silver_policy)} rows")
print(f"Rejected : {len(rejected_policy)} rows")
print(f"\nRate source breakdown:\n{silver_policy.groupby('_rate_source')['period'].agg(['min', 'max', 'count'])}")
print(f"\nSample around 2019 transition:\n{silver_policy[silver_policy['period'].between('2019-04', '2019-08')]}")

Accepted : 197 rows
Rejected : 2160 rows

Rate source breakdown:
                     min      max  count
_rate_source                            
LIBOR_midpoint   2010-01  2019-05    113
SNB_policy_rate  2019-06  2026-05     84

Sample around 2019 transition:
      period  swiss_policy_rate                               _run_id  \
111  2019-04              -0.75  88455a82-f70b-4392-b459-1c2e7a0f0dac   
112  2019-05              -0.75  88455a82-f70b-4392-b459-1c2e7a0f0dac   
113  2019-06              -0.75  88455a82-f70b-4392-b459-1c2e7a0f0dac   
114  2019-07              -0.75  88455a82-f70b-4392-b459-1c2e7a0f0dac   
115  2019-08              -0.75  88455a82-f70b-4392-b459-1c2e7a0f0dac   

                         _ingested_at          _source  \
111  2026-07-15T12:59:47.583578+00:00  snb_policy_rate   
112  2026-07-15T12:59:47.583578+00:00  snb_policy_rate   
113  2026-07-15T12:59:47.583578+00:00  snb_policy_rate   
114  2026-07-15T12:59:47.583578+00:00  snb_policy_rate   
115  2026-

In [9]:
silver_policy.to_parquet(SILVER_PATH / "silver_policy_rate.parquet", index=False)
rejected_policy.to_parquet(SILVER_PATH / "silver_policy_rate_rejected.parquet", index=False)

print(f"Saved silver_policy_rate        : {len(silver_policy)} rows")
print(f"Saved silver_policy_rate_rejected: {len(rejected_policy)} rows")

Saved silver_policy_rate        : 197 rows
Saved silver_policy_rate_rejected: 2160 rows
